**CI twin of `ch18-hierarchical-clustering.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

df = load_csv("penguins").dropna(subset=["flipper_length_mm",
                                         "bill_length_mm"])
Xs = StandardScaler().fit_transform(
    df[["flipper_length_mm", "bill_length_mm"]])

pos6 = []
for sp in ("Adelie", "Chinstrap", "Gentoo"):
    pos6 += [df.index.get_loc(i) for i in df.index[df["species"] == sp][:2]]
pts = Xs[pos6]
D = np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(axis=2))
print("birds 0-1 Adelie, 2-3 Chinstrap, 4-5 Gentoo\n")
print(pd.DataFrame(np.round(D, 2)))

groups = [[i] for i in range(6)]

def group_distance(g1, g2):          # single linkage: nearest members
    return min(D[a][b] for a in g1 for b in g2)

for step in range(1, 4):
    i, j = min(((i, j) for i in range(len(groups))
                for j in range(i + 1, len(groups))),
               key=lambda p: group_distance(groups[p[0]], groups[p[1]]))
    d = group_distance(groups[i], groups[j])
    print(f"\nmerge {step}: {groups[i]} + {groups[j]}  at distance {d:.2f}")
    groups = [g for k, g in enumerate(groups) if k not in (i, j)] \
        + [groups[i] + groups[j]]
print(f"\ngroups now: {groups}")

In [ ]:
from sklearn.cluster import AgglomerativeClustering

rng = np.random.default_rng(0)
sub = rng.choice(len(Xs), 60, replace=False)
X60 = Xs[sub]

for link in ("ward", "single"):
    ac = AgglomerativeClustering(n_clusters=3, linkage=link).fit(X60)
    print(f"{link:7}: cluster sizes {np.bincount(ac.labels_).tolist()}")

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

Z = linkage(X60, method="ward")

fig, ax = plt.subplots(figsize=(7, 3.4))
dendrogram(Z, ax=ax, no_labels=True, color_threshold=4)
ax.axhline(10, ls="--", lw=1, color="grey")
ax.axhline(4, ls="--", lw=1, color="grey")
ax.set_ylabel("merge distance")
plt.show()

for cut in (10, 4):
    labs = fcluster(Z, t=cut, criterion="distance")
    print(f"cut at height {cut}: {len(set(labs))} clusters, "
          f"sizes {np.bincount(labs)[1:].tolist()}")

In [ ]:
ac = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(Xs)
reveal = pd.crosstab(ac.labels_, df["species"])
print(reveal)

agree = reveal.max(axis=1).sum()
print(f"\nagreement: {agree} of {len(df)} ({agree / len(df):.1%})")
print("k-means (Ch17) scored: 95.6%")

In [ ]:
model = AgglomerativeClustering(n_clusters=3, linkage="ward")
model.fit(Xs)

run_tests([
    ("three clusters found", len(set(model.labels_)), 3),
    ("cluster sizes", sorted(np.bincount(model.labels_).tolist()),
     [64, 126, 152]),
])

In [ ]:
import math

def closest_pair(groups, points):
    def gd(g1, g2):
        return min(math.dist(points[a], points[b])
                   for a in g1 for b in g2)
    return min(((i, j) for i in range(len(groups))
                for j in range(i + 1, len(groups))),
               key=lambda p: gd(groups[p[0]], groups[p[1]]))

def merge_step(groups, pair):
    i, j = pair
    rest = [g for k, g in enumerate(groups) if k not in pair]
    return rest + [groups[i] + groups[j]]

pts = [(0.0, 0.0), (1.0, 0.0), (5.0, 5.0), (5.8, 5.0), (10.0, 0.0)]
g0 = [[0], [1], [2], [3], [4]]

run_tests([
    ("first merge: the 0.8-apart pair", closest_pair(g0, pts), (2, 3)),
    ("the diary updates", merge_step(g0, (2, 3)),
     [[0], [1], [4], [2, 3]]),
    ("second merge: the 1.0-apart pair",
     closest_pair([[0], [1], [4], [2, 3]], pts), (0, 1)),
    ("group distance uses nearest members",
     closest_pair([[0, 1], [4], [2, 3]], pts), (0, 2)),
])